In [1]:
import torch
import numpy as np
import matplotlib as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.transforms import v2
import os


In [2]:
print(os.getcwd())

/Users/donniexie/Desktop/xdy_study_cs


### 初始化训练集和测试集

In [3]:
Training_data = datasets.MNIST(
    root="data", #数据储存地址
    train=True, #训练集还是测试集
    download=True, #是否下载数据
    transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale=True)])
)


In [4]:
Test_data = datasets.MNIST(
    root = "data",
    train = False,
    download = True,
    transform = v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32, scale=True)])
)

### 加载数据

In [5]:
batch_size = 32

train_dataloader = DataLoader(
    Training_data,
    shuffle= True,
    batch_size = batch_size
)

test_dataloader = DataLoader(
    Test_data,
    shuffle= True,
    batch_size = batch_size
)

In [6]:
for X, y in train_dataloader:
    print(f"Shape of X [N,C,H,W]:{X.shape}")
    print(f"Shape of y:{y.shape}")
    break

Shape of X [N,C,H,W]:torch.Size([32, 1, 28, 28])
Shape of y:torch.Size([32])


### 创建训练模型

In [7]:
device = "cpu"

class NeuroNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 512), 
            nn.ReLU(),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits
    
model = NeuroNetwork().to(device)
print(model)



NeuroNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [8]:
print(list(model.parameters()))


[Parameter containing:
tensor([[-0.0350, -0.0100,  0.0249,  ...,  0.0003,  0.0327, -0.0177],
        [-0.0008, -0.0006,  0.0065,  ...,  0.0129, -0.0110,  0.0014],
        [ 0.0077, -0.0252, -0.0203,  ...,  0.0255,  0.0014,  0.0108],
        ...,
        [-0.0342, -0.0139,  0.0173,  ..., -0.0012,  0.0078,  0.0245],
        [-0.0105, -0.0053,  0.0265,  ...,  0.0090, -0.0273, -0.0071],
        [-0.0195, -0.0315, -0.0143,  ...,  0.0107,  0.0243, -0.0213]],
       requires_grad=True), Parameter containing:
tensor([-0.0343, -0.0190, -0.0109,  0.0260, -0.0337, -0.0251,  0.0354, -0.0076,
         0.0020, -0.0204,  0.0011, -0.0014,  0.0059,  0.0190, -0.0337, -0.0118,
        -0.0307,  0.0051,  0.0356,  0.0200,  0.0112, -0.0057, -0.0117, -0.0199,
         0.0325, -0.0036,  0.0141,  0.0349,  0.0030, -0.0037,  0.0109,  0.0258,
        -0.0170, -0.0251, -0.0250,  0.0129,  0.0102,  0.0038, -0.0183, -0.0099,
         0.0068,  0.0276, -0.0223,  0.0083,  0.0019,  0.0258, -0.0193, -0.0116,
        -0.02

### 创建优化器

In [9]:
lost_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr = 1e-3)


In [60]:
print(model.train())
print(model(X).shape)

NeuroNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=10, bias=True)
  )
)
torch.Size([32, 10])


In [61]:
print(model.eval())


NeuroNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=512, bias=True)
    (5): ReLU()
    (6): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [10]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

In [ ]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [65]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, lost_fn, optimizer)
    test(test_dataloader, model, lost_fn)

Epoch 1
-------------------------------
loss: 2.258379 [    0/60000]
loss: 2.239875 [ 3200/60000]
loss: 2.269280 [ 6400/60000]
loss: 2.246236 [ 9600/60000]
loss: 2.232258 [12800/60000]
loss: 2.258705 [16000/60000]
loss: 2.239815 [19200/60000]
loss: 2.237633 [22400/60000]
loss: 2.237086 [25600/60000]
loss: 2.229323 [28800/60000]
loss: 2.242397 [32000/60000]
loss: 2.237726 [35200/60000]
loss: 2.234740 [38400/60000]
loss: 2.218884 [41600/60000]
loss: 2.220654 [44800/60000]
loss: 2.195974 [48000/60000]
loss: 2.211844 [51200/60000]
loss: 2.189945 [54400/60000]
loss: 2.178206 [57600/60000]
Test Error: 
 Accuracy: 46.7%, Avg loss: 2.183763 

Epoch 2
-------------------------------
loss: 2.154853 [    0/60000]
loss: 2.175497 [ 3200/60000]
loss: 2.152467 [ 6400/60000]
loss: 2.179895 [ 9600/60000]
loss: 2.149752 [12800/60000]
loss: 2.158404 [16000/60000]
loss: 2.123465 [19200/60000]
loss: 2.101586 [22400/60000]
loss: 2.141011 [25600/60000]
loss: 2.095828 [28800/60000]
loss: 2.062931 [32000/60000

In [ ]:
torch.save(model.state_dict(), "models/model.pth")

In [79]:
classes = [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9"
] 

model.eval()
counter = 0
k = 35
for n in range(k):
    x,y = Test_data[n][0], Test_data[n][1]
    with torch.no_grad():
        x = x.to(device)
        pred = model(x)
        predicted, actual = classes[pred[0].argmax(0)], classes[y]
        print(f'Predicted: "{predicted}", Actual: "{actual}"')
        if predicted == actual:
            counter += 1
print(f"Accuracy: {counter/k*100}%")

Predicted: "7", Actual: "7"
Predicted: "2", Actual: "2"
Predicted: "1", Actual: "1"
Predicted: "0", Actual: "0"
Predicted: "4", Actual: "4"
Predicted: "1", Actual: "1"
Predicted: "9", Actual: "4"
Predicted: "9", Actual: "9"
Predicted: "6", Actual: "5"
Predicted: "9", Actual: "9"
Predicted: "0", Actual: "0"
Predicted: "2", Actual: "6"
Predicted: "9", Actual: "9"
Predicted: "0", Actual: "0"
Predicted: "1", Actual: "1"
Predicted: "3", Actual: "5"
Predicted: "9", Actual: "9"
Predicted: "7", Actual: "7"
Predicted: "2", Actual: "3"
Predicted: "4", Actual: "4"
Predicted: "7", Actual: "9"
Predicted: "6", Actual: "6"
Predicted: "6", Actual: "6"
Predicted: "5", Actual: "5"
Predicted: "4", Actual: "4"
Predicted: "0", Actual: "0"
Predicted: "7", Actual: "7"
Predicted: "4", Actual: "4"
Predicted: "0", Actual: "0"
Predicted: "1", Actual: "1"
Predicted: "3", Actual: "3"
Predicted: "1", Actual: "1"
Predicted: "3", Actual: "3"
Predicted: "6", Actual: "4"
Predicted: "7", Actual: "7"
Accuracy: 80.0%
